Usar a PySUS 1.0.1

In [0]:
# Usar a biblioteca PySUS 1.0.1
import os
import gc
import sys
import shutil
import pandas as pd

from datetime               import date, datetime
from pysus.online_data.SIH  import SIH
from pyspark.sql            import functions as F

In [0]:
# --- Definições Globais do Catálogo no Databricks ---
catalogo = "workspace"
esquema  = "pysus"

In [0]:
# No Databricks, usamos o diretório local do Driver '/tmp/' para o download temporário do PySUS
caminho_download = "/tmp/sih_downloads"

In [0]:
os.makedirs(caminho_download, exist_ok=True)

In [0]:
def criar_tabela_log(catalogo, esquema):
    """Cria a tabela de logs em formato Delta Lake se não existir."""
    
    tabela_log = f"{catalogo}.{esquema}.log_cargas"
    
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabela_log} (
        origem STRING NOT NULL,
        grupo STRING NOT NULL,
        uf STRING NOT NULL,
        ano INT NOT NULL,
        qt_registros BIGINT NOT NULL,
        data_carga TIMESTAMP NOT NULL
    )
    USING DELTA
    """)

In [0]:
def verifica_arquivo_processado(catalogo, esquema, modulo, grupo, uf, ano):
    """Verifica na tabela de logs se a combinação de grupo, uf e ano já foi carregada."""
    
    tabela_log = f"{catalogo}.{esquema}.log_cargas"
    
    # Se a tabela de logs ainda não existe (primeira execução absoluta), não foi processado
    if not spark.catalog.tableExists(tabela_log):
        return 0
        
    df_log = spark.table(tabela_log)
    total = df_log.filter(
        (F.col("grupo")     == grupo    ) & 
        (F.col("uf")        == uf       ) & 
        (F.col("ano")       == int(ano) )
    ).count()
    
    return total

In [0]:
def grava_log(modulo, grupo, uf, ano, qt_registros, catalogo, esquema):
    """Registra uma linha de histórico de carga na tabela Delta de log."""

    tabela_log = f"{catalogo}.{esquema}.log_cargas"
    
    dados_log = [(modulo, grupo, uf, int(ano), int(qt_registros), datetime.now())]
    
    esquema_log = "origem STRING, grupo STRING, uf STRING, ano INT, qt_registros BIGINT, data_carga TIMESTAMP"
    
    df_novo_log = spark.createDataFrame(dados_log, schema=esquema_log)
    
    df_novo_log.write.format("delta").mode("append").saveAsTable(tabela_log)

In [0]:
def limpa_diretorio_download(caminho):
    """Remove os resíduos locais de download para evitar leituras duplicadas no loop."""
    
    if os.path.exists(caminho):
        shutil.rmtree(caminho)
    
    os.makedirs(caminho, exist_ok=True)

In [0]:
sih = SIH().load()

In [0]:
# Obtém a lista dos grupos e extrai as siglas
lista_grupos = list(sih.groups.keys())

# Lista de todas as UFs
ufs = [
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 
    'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 
    'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
]

# Faixa de pesquisa (conforme o arquivo original 2025 a 2026)
anos = list(range(2025, date.today().year))

In [0]:
# Inicializa a tabela de log, se necessário
criar_tabela_log(catalogo, esquema)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Laço para baixar os arquivos por ano, UF e grupo
for grupo in lista_grupos:

    for uf in ufs:

        for ano in anos:

            try:
                tabela_destino = f"{catalogo}.{esquema}.bronze_sih_{grupo}"

                # 1. Validação de Idempotência
                if verifica_arquivo_processado(catalogo, esquema, 'SIH', grupo, uf, ano) == 0:

                    # Busca o arquivo específico no DATASUS
                    arquivos = sih.get_files(grupo, uf=uf, year=ano)

                    if not arquivos:
                        continue

                    # Baixa os arquivos no disco local do Driver (.dbc -> Parquet via PySUS)
                    sih.download(arquivos, local_dir=caminho_download)

                    # Se houver arquivos baixados no diretório, inicia o tratamento
                    if os.path.exists(caminho_download) and len(os.listdir(caminho_download)) > 0:

                        print(f"Processando o grupo: {grupo} da {uf} e do ano {ano}...")

                        # Carrega os dados gerados via pandas
                        df_pandas = pd.read_parquet(caminho_download)
                        
                        # Converte para Spark DataFrame para ganho de escala
                        df_spark = spark.createDataFrame(df_pandas)

                        # Adiciona campos de controle usando funções Spark (Natividade e Performance)
                        df_spark = df_spark.withColumn("uf_origem", F.lit(uf)) \
                                           .withColumn("ano_origem", F.lit(int(ano))) \
                                           .withColumn("data_ingestao", F.current_timestamp())

                        # Reordenar para colocar as novas colunas no início
                        cols_novas      = ["uf_origem", "ano_origem", "data_ingestao"]
                        cols_restantes  = [c for c in df_spark.columns if c not in cols_novas]

                        df_spark        = df_spark.select(cols_novas + cols_restantes)

                        # 2. Tratamento de duplicidade na tabela destino 
                        # Se a tabela já existia e continha dados antigos dessa UF/Ano, removemos via Spark SQL
                        if spark.catalog.tableExists(tabela_destino):
                            spark.sql(f"""
                                DELETE FROM
                                    {tabela_destino} 
                                WHERE
                                    uf_origem   = '{uf}' AND
                                    ano_origem  = {ano}
                            """)

                        print(f"\tInserindo dados no Delta Lake: {tabela_destino}")

                        # 3. Gravação com evolução de esquema automática
                        # Substitui: df_dados.to_sql, verificar_coluna, criar_tabela e loops de INSERT INTO ?
                        (df_spark.write
                            .format("delta")
                            .mode("append")
                            #.option("mergeSchema", "true")
                            .saveAsTable(tabela_destino)
                        )

                        # 4. Auditoria da carga
                        qtd_linhas = df_spark.count()
                        grava_log('SIH', grupo, uf, ano, qtd_linhas, catalogo, esquema)

                        # Limpeza explícita de memória do Driver
                        del df_pandas
                        del df_spark
                        gc.collect()

                        # Remove arquivos locais para a próxima iteração
                        limpa_diretorio_download(caminho_download)

            except Exception as e:

                print(f"Erro em {grupo} - {uf}/{ano} - Mensagem: {e}")

                # No Databricks, sys.exit(0) mata o container/processo do driver. 
                # É recomendado lançar o erro (raise) para alertar o orquestrador (Workflows).
                raise e

In [0]:
limpa_diretorio_download(caminho_download)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:503)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:794)
	at com.data